In [4]:

# ============================================================
# 1. IMPORTS AND CONFIGURATION
# ============================================================

import time
import re
import json
import joblib
import numpy as np
import pandas as pd

from pathlib import Path
from itertools import product

from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from gensim.corpora import Dictionary
from gensim.models import CoherenceModel


# Folder where results will be saved
OUTPUT_DIR = Path("results/lda")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Five-year benchmark period
BENCHMARK_START_YEAR = 2019
BENCHMARK_END_YEAR = 2023

# Train on 2019-2022 and test on 2023
TEST_YEAR = 2023

# Columns containing different text representations
TEXT_COLUMNS = [
    "speech_no_custom_stopwords",
    "speech_lemmatized",
    "speech_stemmed",
]

# Number of topics to compare
N_TOPICS_OPTIONS = [5, 10, 15, 20]

# Minimum document frequency for terms
MIN_DF_OPTIONS = [2, 5]

# Compare individual unigrams, individual bigrams and both
NGRAM_OPTIONS = [
    (1, 1),
    (2, 2),
    (1, 2),
]

# Compare count-based and TF-IDF representations
VECTORIZER_OPTIONS = ["count", "tfidf"]

# LDA configuration
MAX_ITER = 10
RANDOM_STATE = 42
TOP_WORDS = 10


ModuleNotFoundError: No module named 'gensim'

In [ ]:
# ============================================================
# 2. CHECK AND PREPARE THE DATA
# ============================================================

# Check that the required columns exist
required_columns = ["date"] + TEXT_COLUMNS

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing columns in df: {missing_columns}"
    )

# Work on a copy to preserve the original DataFrame
data = df.copy()

# Convert dates to pandas datetime format
data["date"] = pd.to_datetime(
    data["date"],
    errors="coerce",
)

# Extract year
data["year"] = data["date"].dt.year

# Keep documents from the five-year benchmark period
benchmark_data = data[
    data["year"].between(
        BENCHMARK_START_YEAR,
        BENCHMARK_END_YEAR,
    )
].copy()

# Chronological train/test split
train_data = benchmark_data[
    benchmark_data["year"] < TEST_YEAR
].copy()

test_data = benchmark_data[
    benchmark_data["year"] == TEST_YEAR
].copy()

print("BENCHMARK DATA")
print("-" * 50)
print(f"Period: {BENCHMARK_START_YEAR}-{BENCHMARK_END_YEAR}")
print(f"Training documents: {len(train_data):,}")
print(f"Test documents: {len(test_data):,}")
print(
    "Training years:",
    sorted(train_data["year"].dropna().unique()),
)
print(
    "Test years:",
    sorted(test_data["year"].dropna().unique()),
)

if train_data.empty or test_data.empty:
    raise ValueError(
        "The training or test dataset is empty. "
        "Check the dates in the 'date' column."
    )



In [ ]:
# ============================================================
# 3. HELPER FUNCTIONS
# ============================================================

def prepare_text(series):
    """
    Convert a text column into a list of strings.
    Missing or empty texts are removed.
    """
    texts = series.fillna("").astype(str).str.strip()

    # Remove empty strings
    texts = texts[texts != ""]

    return texts.tolist()


def create_vectorizer(
    vectorizer_type,
    ngram_range,
    min_df,
):
    """
    Create the selected vectorizer.

    CountVectorizer represents documents using word counts.
    TfidfVectorizer represents documents using TF-IDF weights.
    """
    common_parameters = {
        "ngram_range": ngram_range,
        "min_df": min_df,
        "max_df": 0.95,
        "max_features": 10000,
        "lowercase": True,
        "token_pattern": r"(?u)\b\w+\b",
    }

    if vectorizer_type == "count":
        return CountVectorizer(**common_parameters)

    if vectorizer_type == "tfidf":
        return TfidfVectorizer(**common_parameters)

    raise ValueError(
        f"Unknown vectorizer: {vectorizer_type}"
    )


def get_topic_words(model, vectorizer, n_words=10):
    """
    Return the most important words for each topic.
    """
    feature_names = vectorizer.get_feature_names_out()

    topics = []

    for topic_weights in model.components_:
        top_indices = topic_weights.argsort()[-n_words:][::-1]

        topic_words = [
            feature_names[index]
            for index in top_indices
        ]

        topics.append(topic_words)

    return topics


def calculate_topic_diversity(topics):
    """
    Calculate topic diversity.

    Diversity = unique top words across topics /
                total top-word positions.

    A higher value indicates less repetition between topics.
    """
    all_words = [
        word
        for topic in topics
        for word in topic
    ]

    if not all_words:
        return np.nan

    return len(set(all_words)) / len(all_words)


def calculate_coherence(
    model,
    vectorizer,
    training_texts,
    n_words=10,
):
    """
    Calculate C_v topic coherence using training documents.

    The vectorizer's analyzer is used to generate tokens so
    that bigrams are represented consistently with its vocabulary.
    """
    topics = get_topic_words(
        model,
        vectorizer,
        n_words=n_words,
    )

    analyzer = vectorizer.build_analyzer()

    # Gensim represents a bigram such as "migration policy"
    # as "migration_policy".
    coherence_texts = [
        [
            token.replace(" ", "_")
            for token in analyzer(text)
        ]
        for text in training_texts
    ]

    dictionary = Dictionary(coherence_texts)

    if len(dictionary) == 0:
        return np.nan

    # Convert topic terms to the same representation as the tokens
    coherence_topics = [
        [
            word.replace(" ", "_")
            for word in topic
            if word.replace(" ", "_") in dictionary.token2id
        ]
        for topic in topics
    ]

    # Exclude topics with no terms in the coherence dictionary
    coherence_topics = [
        topic for topic in coherence_topics if topic
    ]

    if not coherence_topics:
        return np.nan

    coherence_model = CoherenceModel(
        topics=coherence_topics,
        texts=coherence_texts,
        dictionary=dictionary,
        coherence="c_v",
        processes=1,
    )

    return coherence_model.get_coherence()

In [ ]:
# ============================================================
# 4. BENCHMARK DIFFERENT LDA CONFIGURATIONS
# ============================================================

benchmark_results = []

# Each configuration uses:
# - one text representation
# - one vectorizer
# - one n-gram range
# - one minimum document frequency
# - one number of topics

configurations = list(
    product(
        TEXT_COLUMNS,
        VECTORIZER_OPTIONS,
        NGRAM_OPTIONS,
        MIN_DF_OPTIONS,
        N_TOPICS_OPTIONS,
    )
)

total_configurations = len(configurations)

print("\nSTARTING LDA BENCHMARK")
print("-" * 50)
print(f"Configurations to evaluate: {total_configurations}")


for config_number, configuration in enumerate(
    configurations,
    start=1,
):
    (
        text_column,
        vectorizer_type,
        ngram_range,
        min_df,
        n_topics,
    ) = configuration

    print(
        f"\n[{config_number}/{total_configurations}] "
        f"Text={text_column} | "
        f"Vectorizer={vectorizer_type} | "
        f"N-grams={ngram_range} | "
        f"min_df={min_df} | "
        f"Topics={n_topics}"
    )

    # --------------------------------------------------------
    # 4.1 Prepare training and test texts
    # --------------------------------------------------------

    train_texts = prepare_text(
        train_data[text_column]
    )

    test_texts = prepare_text(
        test_data[text_column]
    )

    if not train_texts or not test_texts:
        print("Skipping: empty training or test texts.")
        continue

    # --------------------------------------------------------
    # 4.2 Fit vectorizer ONLY on the training data
    # --------------------------------------------------------

    vectorizer = create_vectorizer(
        vectorizer_type=vectorizer_type,
        ngram_range=ngram_range,
        min_df=min_df,
    )

    try:
        X_train = vectorizer.fit_transform(train_texts)

        # Transform test data using the training vocabulary
        X_test = vectorizer.transform(test_texts)

    except ValueError as error:
        print(f"Skipping configuration: {error}")
        continue

    # Skip configurations with no usable features
    if X_train.shape[1] == 0:
        print("Skipping: no features were generated.")
        continue

    # --------------------------------------------------------
    # 4.3 Train the LDA model
    # --------------------------------------------------------

    lda = LatentDirichletAllocation(
        n_components=n_topics,
        max_iter=MAX_ITER,
        learning_method="batch",
        random_state=RANDOM_STATE,
        evaluate_every=-1,
        n_jobs=-1,
    )

    start_time = time.time()

    lda.fit(X_train)

    training_seconds = time.time() - start_time

    # --------------------------------------------------------
    # 4.4 Calculate topics and coherence
    # --------------------------------------------------------

    topics = get_topic_words(
        lda,
        vectorizer,
        n_words=TOP_WORDS,
    )

    topic_diversity = calculate_topic_diversity(
        topics
    )

    coherence = calculate_coherence(
        model=lda,
        vectorizer=vectorizer,
        training_texts=train_texts,
        n_words=TOP_WORDS,
    )

    # --------------------------------------------------------
    # 4.5 Calculate held-out metrics
    # --------------------------------------------------------

    # Log-likelihood and perplexity are recorded only for
    # CountVectorizer, where the input represents word counts.
    #
    # Do not directly compare these metrics with TF-IDF results.

    test_log_likelihood = np.nan
    test_perplexity = np.nan

    if vectorizer_type == "count":
        test_log_likelihood = lda.score(X_test)
        test_perplexity = lda.perplexity(X_test)

    # --------------------------------------------------------
    # 4.6 Store results
    # --------------------------------------------------------

    result = {
        "text_column": text_column,
        "vectorizer": vectorizer_type,
        "ngram_range": str(ngram_range),
        "min_df": min_df,
        "n_topics": n_topics,
        "n_features": X_train.shape[1],
        "n_train_documents": X_train.shape[0],
        "n_test_documents": X_test.shape[0],
        "coherence_cv": coherence,
        "topic_diversity": topic_diversity,
        "test_log_likelihood": test_log_likelihood,
        "test_perplexity": test_perplexity,
        "training_seconds": training_seconds,
        "topics": json.dumps(topics, ensure_ascii=False),
    }

    benchmark_results.append(result)

    print(
        f"Coherence: {coherence:.4f} | "
        f"Diversity: {topic_diversity:.4f} | "
        f"Time: {training_seconds:.1f}s"
    )


In [ ]:
# ============================================================
# 5. SAVE AND RANK BENCHMARK RESULTS
# ============================================================

results_df = pd.DataFrame(benchmark_results)

if results_df.empty:
    raise RuntimeError(
        "No configurations completed successfully. "
        "Check the input data and vectorizer settings."
    )

benchmark_path = OUTPUT_DIR / "lda_benchmark_results.csv"

results_df.to_csv(
    benchmark_path,
    index=False,
)

# Rank primarily by coherence and secondarily by diversity.
# Perplexity is not used to rank CountVectorizer against TF-IDF.
results_df = results_df.sort_values(
    by=["coherence_cv", "topic_diversity"],
    ascending=[False, False],
    na_position="last",
).reset_index(drop=True)

ranked_path = OUTPUT_DIR / "lda_benchmark_ranked.csv"

results_df.to_csv(
    ranked_path,
    index=False,
)

print("\nTOP 10 CONFIGURATIONS")
print("-" * 50)

display_columns = [
    "text_column",
    "vectorizer",
    "ngram_range",
    "min_df",
    "n_topics",
    "coherence_cv",
    "topic_diversity",
    "test_perplexity",
    "training_seconds",
]

print(
    results_df[display_columns]
    .head(10)
    .to_string(index=False)
)

print(f"\nBenchmark results saved to: {ranked_path}")


In [ ]:
# ============================================================
# 6. SELECT THE BEST CONFIGURATION
# ============================================================

# Select the highest-ranked configuration with a valid
# coherence score.
valid_results = results_df.dropna(
    subset=["coherence_cv"]
)

if valid_results.empty:
    raise RuntimeError(
        "No valid coherence scores were obtained."
    )

best_result = valid_results.iloc[0]

best_text_column = best_result["text_column"]
best_vectorizer_type = best_result["vectorizer"]
best_ngram_range = tuple(
    json.loads(
        best_result["ngram_range"].replace("(", "[").replace(")", "]")
    )
)
best_min_df = int(best_result["min_df"])
best_n_topics = int(best_result["n_topics"])

print("\nSELECTED CONFIGURATION")
print("-" * 50)
print(f"Text column: {best_text_column}")
print(f"Vectorizer: {best_vectorizer_type}")
print(f"N-gram range: {best_ngram_range}")
print(f"min_df: {best_min_df}")
print(f"Number of topics: {best_n_topics}")
print(f"Benchmark coherence: {best_result['coherence_cv']:.4f}")

In [ ]:
# ============================================================
# 7. TRAIN THE FINAL MODEL ON THE COMPLETE DATASET
# ============================================================

# IMPORTANT:
# The final model is fitted on all available documents,
# not just the five-year benchmark period.

final_texts = prepare_text(
    data[best_text_column]
)

# Keep track of the original row indices corresponding to
# documents that have non-empty text.
final_text_series = (
    data[best_text_column]
    .fillna("")
    .astype(str)
    .str.strip()
)

valid_text_mask = final_text_series != ""

final_document_indices = data.index[valid_text_mask]

final_texts = final_text_series.loc[
    final_document_indices
].tolist()

print("\nTRAINING FINAL MODEL")
print("-" * 50)
print(f"Documents used: {len(final_texts):,}")

# Fit the chosen vectorizer on the full corpus
final_vectorizer = create_vectorizer(
    vectorizer_type=best_vectorizer_type,
    ngram_range=best_ngram_range,
    min_df=best_min_df,
)

X_full = final_vectorizer.fit_transform(
    final_texts
)

# Train a new LDA model using all available documents
final_lda = LatentDirichletAllocation(
    n_components=best_n_topics,
    max_iter=MAX_ITER,
    learning_method="batch",
    random_state=RANDOM_STATE,
    evaluate_every=-1,
    n_jobs=-1,
)

final_lda.fit(X_full)

# Extract final topics
final_topics = get_topic_words(
    final_lda,
    final_vectorizer,
    n_words=TOP_WORDS,
)

print("\nFINAL TOPICS")
print("-" * 50)

for topic_number, words in enumerate(
    final_topics,
    start=1,
):
    print(f"Topic {topic_number}: {', '.join(words)}")

In [ ]:
# ============================================================
# 8. SAVE THE FINAL MODEL AND VECTORIZER
# ============================================================

model_path = OUTPUT_DIR / "lda_final_model.joblib"
vectorizer_path = OUTPUT_DIR / "lda_final_vectorizer.joblib"
topics_path = OUTPUT_DIR / "lda_final_topics.json"
config_path = OUTPUT_DIR / "lda_final_config.json"

joblib.dump(
    final_lda,
    model_path,
)

joblib.dump(
    final_vectorizer,
    vectorizer_path,
)

with open(topics_path, "w", encoding="utf-8") as file:
    json.dump(
        {
            f"topic_{i + 1}": words
            for i, words in enumerate(final_topics)
        },
        file,
        ensure_ascii=False,
        indent=4,
    )

final_config = {
    "benchmark_start_year": BENCHMARK_START_YEAR,
    "benchmark_end_year": BENCHMARK_END_YEAR,
    "test_year": TEST_YEAR,
    "final_training_documents": len(final_texts),
    "text_column": best_text_column,
    "vectorizer": best_vectorizer_type,
    "ngram_range": list(best_ngram_range),
    "min_df": best_min_df,
    "n_topics": best_n_topics,
    "max_iter": MAX_ITER,
    "random_state": RANDOM_STATE,
    "benchmark_coherence_cv": float(
        best_result["coherence_cv"]
    ),
}

with open(config_path, "w", encoding="utf-8") as file:
    json.dump(
        final_config,
        file,
        ensure_ascii=False,
        indent=4,
    )

In [ ]:
# ============================================================
# 9. CALCULATE TOPIC DISTRIBUTIONS FOR ALL DOCUMENTS
# ============================================================

# Each row contains the probability of each topic for one speech.
# A document can have probabilities distributed across several topics.

topic_distributions = final_lda.transform(X_full)

topic_columns = [
    f"topic_{i + 1}"
    for i in range(best_n_topics)
]

topic_df = pd.DataFrame(
    topic_distributions,
    columns=topic_columns,
    index=final_document_indices,
)

# Combine topic probabilities with the original metadata
metadata_columns = [
    column
    for column in [
        "doc_id",
        "date",
        "party_name",
        "speech_en",
    ]
    if column in data.columns
]

document_topics = data.loc[
    final_document_indices,
    metadata_columns,
].copy()

document_topics = document_topics.join(
    topic_df,
)

# Assign the most probable topic to each document
document_topics["dominant_topic"] = (
    topic_df.idxmax(axis=1)
)

document_topics["dominant_topic_probability"] = (
    topic_df.max(axis=1)
)

document_topics["dominant_topic"] = (
    document_topics["dominant_topic"]
    .str.replace("topic_", "", regex=False)
    .astype(int)
)

document_topics_path = (
    OUTPUT_DIR / "lda_document_topic_distributions.csv"
)

document_topics.to_csv(
    document_topics_path,
    index=False,
)

print("\nPROCESS COMPLETE")
print("-" * 50)
print(f"Benchmark results: {ranked_path}")
print(f"Final model: {model_path}")
print(f"Final vectorizer: {vectorizer_path}")
print(f"Topic definitions: {topics_path}")
print(f"Final configuration: {config_path}")
print(f"Document-topic distributions: {document_topics_path}")
